### T-PHATE Embedding Visualization

In [ ]:
# If needed (run once):
# !pip install tphate scprep graphtools
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import tphate

In [ ]:
def load_run_tsv(tsv_path: str | Path) -> np.ndarray:
    """
    Loads a parcel time-series TSV and returns X with shape (T, P).
    Keeps numeric columns only; drops rows with NaNs.
    """
    df = pd.read_csv(tsv_path, sep="\t")
    df_num = df.select_dtypes(include=[np.number]).copy()
    df_num = df_num.dropna(axis=1, how="all").dropna(axis=0, how="any")
    X = df_num.to_numpy(dtype=np.float32)
    if X.ndim != 2 or X.shape[0] <= 10 or X.shape[1] <= 1:
        raise ValueError(f"Bad shape from {tsv_path}: {X.shape}. Expected (T, P).")
    return X

def find_run_file(subject_dir: Path, run_key: str) -> Path:
    """
    Finds the TSV for a given run inside subject_dir.
    Customize the pattern to match your filenames.
    """
    hits = list(subject_dir.glob(f"*{run_key}*.tsv"))
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected 1 file for run '{run_key}' in {subject_dir}, found {len(hits)}")
    return hits[0]

In [ ]:
DATA_ROOT = Path("/path/to/data/derivatives/parcellated_cropped")
subject_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith("sub-")])
runs = ["AntiLeft", "AntiRight", "ProLeft", "ProRight"]

run_files = {}
Xs = {}

# ---- load all subjects x runs ----
for subject_dir in subject_dirs:
    subj_id = subject_dir.name
    run_files[subj_id] = {run: find_run_file(subject_dir, run) for run in runs}
    Xs[subj_id] = {run: load_run_tsv(run_files[subj_id][run]) for run in runs}

# ---- quick sanity print ----
for subj_id in Xs:
    for run in runs:
        f = run_files[subj_id][run]
        X = Xs[subj_id][run]
        print(subj_id, Path(f).name, "->", X.shape, "(T, P)")


In [ ]:
# ---- T-PHATE params ----
tph_params = dict(
    n_components=2,
    knn=10,
    t=20,
    random_state=0,   # recommended for stability
    verbose=False
)

# ---- embed: embeddings[run][subj_id] = Y ----
embeddings = {run: {} for run in runs}

for subj_id in Xs:
    for run in runs:
        X = Xs[subj_id][run]
        Y = tphate.TPHATE(**tph_params).fit_transform(X)
        embeddings[run][subj_id] = Y

print("Done. Example counts per run:", {run: len(embeddings[run]) for run in runs})


In [ ]:
# plot one subject - 4 runs
example_subj = subject_dirs[0].name
Ys = [embeddings[run][example_subj] for run in runs]
example_run_files = [run_files[example_subj][run] for run in runs]  # don't overwrite run_files
run_colors = {
    "AntiLeft": "red",
    "AntiRight": "blue",
    "ProLeft": "green",
    "ProRight": "orange",
}
for run, f, Y in zip(runs, example_run_files, Ys):
    color = run_colors.get(run, "gray")
    plt.figure()
    plt.scatter(Y[:, 0], Y[:, 1], color=color, alpha=0.7, s=12)
    plt.xlabel("T-PHATE 1")
    plt.ylabel("T-PHATE 2")
    plt.title(f"T-PHATE (parcels) — {example_subj}, run: {run}")
    plt.show()
# # ---- plot: one per run, colored by time ----
# for f, Y in zip(run_files, Ys):
#     t_idx = np.arange(Y.shape[0])

#     plt.figure()
#     plt.scatter(Y[:, 0], Y[:, 1], c=t_idx)
#     plt.colorbar(label="time (TR index)")
#     plt.xlabel("T-PHATE 1")
#     plt.ylabel("T-PHATE 2")
#     plt.title(f"T-PHATE (parcels) — {Path(f).name}")
#     plt.show()

In [ ]:
# ---- plot: panels = runs, lines = subjects ----
n_runs = len(runs)
fig, axes = plt.subplots(1, n_runs, figsize=(5 * n_runs, 5), squeeze=False)
axes = axes[0]

for ax, run in zip(axes, runs):
    for subj_id, Y in embeddings[run].items():
        ax.plot(Y[:, 0], Y[:, 1], alpha=0.25, linewidth=1)
    ax.set_title(run)
    ax.set_xlabel("T-PHATE 1")
    ax.set_ylabel("T-PHATE 2")

plt.tight_layout()
plt.show()